# BRIDGE data audit (revision, Reviewers 3 and 4)

Runs **no training**. Loads the four datasets with the same loaders as the main notebook and reports:

* **A.** raw file order, timestamp / flow-ID columns, label runs (can real chronological sequences be built?)
* **B.** units of every mapped feature, with consistency checks computed from the data
* **C.** window-label ties and purity for W = 31, 32, 33
* **D.** exact-duplicate counts across train / validation / test and across datasets

Attach the same four Kaggle datasets as the main notebook, choose **CPU** (no GPU needed), and *Save Version → Save & Run All*.
When it finishes, download `bridge_audit/bridge_audit.json` from the Output tab and send it back.

In [ ]:
import os, glob, json, time, warnings, copy, gc, itertools, math
warnings.filterwarnings('ignore')
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import numpy as np
import pandas as pd
import matplotlib
try:
    get_ipython()
    import matplotlib.pyplot as plt
except NameError:
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import matplotlib.ticker as ticker

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    f1_score, accuracy_score, precision_score, recall_score,
    roc_curve, precision_recall_curve, auc, matthews_corrcoef)
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from scipy import stats as scipy_stats

try:
    from xgboost import XGBClassifier; XGB_OK = True
except ImportError:
    XGB_OK = False; print("XGBoost unavailable — fallback to GradientBoosting")

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True      # fastest CUDA kernels
    torch.backends.cudnn.deterministic = False  # benchmark requires non-deterministic

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")
print(f"CUDA: {torch.cuda.is_available()} | XGBoost: {XGB_OK}")


In [ ]:
# ── 46 canonical features ─────────────────────────────────────────────────
SEMANTIC_FEATURES = [
    # ── Flow-level counts & rates (T-branch primary) ─────────────────────
    "flow_duration",        # 0  total flow duration
    "pkt_count_fwd",        # 1  total forward packets
    "pkt_count_bwd",        # 2  total backward packets
    "byte_count_fwd",       # 3  total forward bytes
    "byte_count_bwd",       # 4  total backward bytes
    "pkt_rate",             # 5  packets per second (flow level)
    "byte_rate",            # 6  bytes per second
    "fwd_pkt_rate",         # 7  forward packets per second
    "bwd_pkt_rate",         # 8  backward packets per second
    "fwd_byte_rate",        # 9  forward bytes per second
    "bwd_byte_rate",        # 10 backward bytes per second
    "pkt_count_total",      # 11 total packets (fwd + bwd)
    "byte_count_total",     # 12 total bytes
    "fwd_pkt_len_total",    # 13 total length of fwd packets
    "bwd_pkt_len_total",    # 14 total length of bwd packets
    "subflow_fwd_pkts",     # 15 subflow forward packets
    "subflow_bwd_pkts",     # 16 subflow backward packets
    # ── Packet size statistics (H-branch primary) ────────────────────────
    "pkt_len_min",          # 17
    "pkt_len_max",          # 18
    "pkt_len_mean",         # 19
    "pkt_len_std",          # 20
    "pkt_len_var",          # 21
    "fwd_pkt_len_min",      # 22
    "fwd_pkt_len_max",      # 23
    "fwd_pkt_len_mean",     # 24
    "fwd_pkt_len_std",      # 25
    "bwd_pkt_len_min",      # 26
    "bwd_pkt_len_max",      # 27
    "bwd_pkt_len_mean",     # 28
    "bwd_pkt_len_std",      # 29
    # ── IAT statistics (T + H) ───────────────────────────────────────────
    "iat_mean",             # 30
    "iat_std",              # 31
    "iat_max",              # 32
    "iat_min",              # 33
    "fwd_iat_mean",         # 34
    "fwd_iat_std",          # 35
    "bwd_iat_mean",         # 36
    "bwd_iat_std",          # 37
    # ── TCP flags (H-branch) ─────────────────────────────────────────────
    "flag_syn",             # 38
    "flag_ack",             # 39
    "flag_fin",             # 40
    "flag_rst",             # 41
    "flag_psh",             # 42
    "flag_urg",             # 43
    # ── Header / window ──────────────────────────────────────────────────
    "fwd_header_len",       # 44
    "init_win_fwd",         # 45
]
N_SEM = len(SEMANTIC_FEATURES)
SEM_IDX = {name: i for i, name in enumerate(SEMANTIC_FEATURES)}
print(f"Canonical vocabulary: {N_SEM} features")

# ── Per-dataset alias maps ───────────────────────────────────────────────
# Key = canonical name. Value = list of possible column names (case-insensitive).
# ONLY genuinely equivalent features are mapped. No fabricated proxies.
# Short aliases (<5 chars) NEVER matched via substring — only exact match.
#
# ── DATASET RESET (v7) ────────────────────────────────────────────────────
# CIC-IoT-DIAD-2024 (Mostafa Ali Ogab, Kaggle) was confirmed BROKEN — every
# file (including attack-named files) carries "NeedManualLabel" in the Label
# column. Genuinely unusable for supervised learning. DROPPED.
# CICIoMT-2024 (Lian Mollick, Kaggle) was confirmed WRONG VARIANT — 45 cols,
# no label column at all (same reduced schema as CIC-IoT-2023). DROPPED.
#
# Replaced with, and verified directly against user-supplied sample CSVs:
#   • CICIDS-2017                          (sample: cicids-2017-friday[-plus].csv)
#   • CIC-BCCC-NRC-TabularIoTAttacks-2024   (sample: cic-bccc-nrc-tabulariotAttack.csv)
#   • CICIoMT-2024 (Train-Test, D. Kilichev) (sample: ciciomt-{train,test}-set.csv)

# ── ALIAS_FLOWMETER ──────────────────────────────────────────────────────
# Shared by CICIDS-2017 AND CIC-BCCC-NRC-TabularIoTAttacks-2024.
# Verified column-for-column against both uploaded samples — they use the
# IDENTICAL CICFlowMeter naming convention (BCCC's updated extractor):
#   "Total Fwd Packet"           (singular, no trailing 's')
#   "Total Bwd packets"          (lowercase 'p', trailing 's')
#   "Total Length of Fwd Packet" (singular)
#   "Total Length of Bwd Packet" (singular)
#   "FWD Init Win Bytes"         (upper-case FWD)
# Old verbose CICIDS MachineLearningCSV.zip names ("Total Fwd Packets",
# "Init_Win_bytes_forward", ...) are kept as fallback aliases in case a
# differently-formatted CICIDS-2017 file is substituted later.
# cic-bccc-nrc-tabulariotAttack.csv additionally carries an explicit
# "Attack Name" string column (e.g. "Benign Traffic") used for labeling —
# see load_cic_bccc_nrc() below — it is NOT part of the feature vocabulary.
ALIAS_FLOWMETER = {
    "flow_duration":     ["Flow Duration"],
    "pkt_count_fwd":     ["Total Fwd Packet",              "Total Fwd Packets",
                          "Tot Fwd Pkts"],
    "pkt_count_bwd":     ["Total Bwd packets",             "Total Backward Packets",
                          "Tot Bwd Pkts"],
    "byte_count_fwd":    ["Total Length of Fwd Packet",    "Total Length of Fwd Packets",
                          "TotLen Fwd Pkts"],
    "byte_count_bwd":    ["Total Length of Bwd Packet",    "Total Length of Bwd Packets",
                          "TotLen Bwd Pkts"],
    "pkt_rate":          ["Flow Packets/s",                "Flow Pkts/s"],
    "byte_rate":         ["Flow Bytes/s",                  "Flow Byts/s"],
    "fwd_pkt_rate":      ["Fwd Packets/s",                 "Fwd Pkts/s"],
    "bwd_pkt_rate":      ["Bwd Packets/s",                 "Bwd Pkts/s"],
    "fwd_pkt_len_total": ["Total Length of Fwd Packet",    "Total Length of Fwd Packets",
                          "TotLen Fwd Pkts"],
    "bwd_pkt_len_total": ["Total Length of Bwd Packet",    "Total Length of Bwd Packets",
                          "TotLen Bwd Pkts"],
    "subflow_fwd_pkts":  ["Subflow Fwd Packets",           "Subflow Fwd Pkts"],
    "subflow_bwd_pkts":  ["Subflow Bwd Packets",           "Subflow Bwd Pkts"],
    "pkt_len_min":       ["Packet Length Min",             "Min Packet Length",
                          "Pkt Len Min"],
    "pkt_len_max":       ["Packet Length Max",             "Max Packet Length",
                          "Pkt Len Max"],
    "pkt_len_mean":      ["Packet Length Mean",            "Pkt Len Mean"],
    "pkt_len_std":       ["Packet Length Std",             "Pkt Len Std"],
    "pkt_len_var":       ["Packet Length Variance",        "Pkt Len Var"],
    "fwd_pkt_len_min":   ["Fwd Packet Length Min",         "Fwd Pkt Len Min"],
    "fwd_pkt_len_max":   ["Fwd Packet Length Max",         "Fwd Pkt Len Max"],
    "fwd_pkt_len_mean":  ["Fwd Packet Length Mean",        "Fwd Pkt Len Mean"],
    "fwd_pkt_len_std":   ["Fwd Packet Length Std",         "Fwd Pkt Len Std"],
    "bwd_pkt_len_min":   ["Bwd Packet Length Min",         "Bwd Pkt Len Min"],
    "bwd_pkt_len_max":   ["Bwd Packet Length Max",         "Bwd Pkt Len Max"],
    "bwd_pkt_len_mean":  ["Bwd Packet Length Mean",        "Bwd Pkt Len Mean"],
    "bwd_pkt_len_std":   ["Bwd Packet Length Std",         "Bwd Pkt Len Std"],
    "iat_mean":          ["Flow IAT Mean"],
    "iat_std":           ["Flow IAT Std"],
    "iat_max":           ["Flow IAT Max"],
    "iat_min":           ["Flow IAT Min"],
    "fwd_iat_mean":      ["Fwd IAT Mean"],
    "fwd_iat_std":       ["Fwd IAT Std"],
    "bwd_iat_mean":      ["Bwd IAT Mean"],
    "bwd_iat_std":       ["Bwd IAT Std"],
    "flag_syn":          ["SYN Flag Count",               "SYN Flag Cnt"],
    "flag_ack":          ["ACK Flag Count",               "ACK Flag Cnt"],
    "flag_fin":          ["FIN Flag Count",               "FIN Flag Cnt"],
    "flag_rst":          ["RST Flag Count",               "RST Flag Cnt"],
    "flag_psh":          ["PSH Flag Count",               "PSH Flag Cnt"],
    "flag_urg":          ["URG Flag Count",               "URG Flag Cnt"],
    "fwd_header_len":    ["Fwd Header Length",            "Fwd Header Len"],
    "init_win_fwd":      ["FWD Init Win Bytes",           "Init_Win_bytes_forward",
                          "Init Fwd Win Bytes",           "Init Fwd Win Byts"],
}
# Coverage: 42/46 (91%) — verified via live smoke test directly against both uploaded sample CSVs
# (cicids-2017-friday.csv, cic-bccc-nrc-tabulariotAttack.csv).
# 4 columns present in the CICIDS-2017 sample only (ICMP Code, ICMP Type,
# Total TCP Flow Time, Attempted Category) and 1 present in the BCCC sample
# only (Attack Name) are NOT canonical features and are intentionally unmapped.
# Unmapped canonical features (8): fwd_byte_rate, bwd_byte_rate (no per-direction
# byte-rate column in either dataset), pkt_count_total / byte_count_total
# (derived in combine() from fwd+bwd, not read from a column).

# ── ALIAS_IOMT ───────────────────────────────────────────────────────────
# CICIoMT-2024 (Train-Test split, D. Kilichev re-upload). 46 columns.
# Verified against: ciciomt-train-set.csv, ciciomt-test-set.csv
# SAME 45-feature schema as the a known-broken re-upload variant, but THIS
# variant carries a genuine, populated "Label" column:
#   train sample Label values: "DoS-TCP"
#   test  sample Label values: "Malformed_Data"
# (Dadkhah et al. 2024 — 19 attack-class string labels + benign; benign
# rows are NOT present in either 10-row sample, so the exact benign string
# is unconfirmed — load_ciciomt() below matches ANY label containing
# "benign" case-insensitively, not just an exact string, to stay robust
# to "Benign" / "BenignTraffic" / "Benign Traffic" variants.)
# Coverage: 17/46 (37%) features ALIAS-RESOLVE correctly after the Srate/Drate
# correction below. The EFFECTIVE coverage, measured empirically as "has at
# least one non-zero value across the full loaded dataset" (the stricter,
# more defensible metric — see Cell 6's feature_coverage_report.csv), is
# 16/46 (35%): "Drate" (-> bwd_pkt_rate) resolves to a real column, but that
# column's values are ~0 across virtually every row in this dataset — which
# independently matches the original CICIoT2023 paper's own published stats
# (Drate mean=5.46e-6, median/25/50/75th percentiles all exactly 0). This is
# a genuine property of the data, not a mapping defect. Separately, "Srate"
# (-> fwd_pkt_rate) and "Rate" (-> pkt_rate) were found to be numerically
# IDENTICAL in this dataset (again matching the source paper's stats table
# exactly) — both alias-resolve and both carry signal, but they are
# redundant with each other, not independent features. Report 16/46 (35%)
# as the dataset's effective coverage in the paper, not 17/46 — it is the
# number that will reproduce if a reviewer re-runs this pipeline.
# Protocol-presence flags (HTTP, HTTPS, TCP, UDP etc.) and statistical
# moment features (Magnitue, Radius, Covariance, Weight) have no
# CICFlowMeter equivalents — correctly zero-filled.
ALIAS_IOMT = {
    "pkt_count_total":  ["Number", "number"],
    "byte_count_total": ["Tot sum",  "tot sum",  "Tot size", "tot size"],
    "pkt_rate":         ["Rate",     "rate"],
    "fwd_pkt_rate":     ["Srate",    "srate"],   # see note below
    "bwd_pkt_rate":     ["Drate",    "drate"],   # see note below
    "pkt_len_min":      ["Min",      "min"],
    "pkt_len_max":      ["Max",      "max"],
    "pkt_len_mean":     ["AVG",      "avg"],
    "pkt_len_std":      ["Std",      "std"],
    "pkt_len_var":      ["Variance", "variance"],
    "iat_mean":         ["IAT",      "iat"],
    "flag_syn":         ["syn_flag_number", "syn_count"],
    "flag_ack":         ["ack_flag_number", "ack_count"],
    "flag_fin":         ["fin_flag_number", "fin_count"],
    "flag_rst":         ["rst_flag_number", "rst_count"],
    "flag_psh":         ["psh_flag_number"],
    "fwd_header_len":   ["Header_Length",   "header_length"],
}
# Srate / Drate added after cross-checking against the CICIoT2023 extractor's
# own published feature definitions (Neto et al. 2023, Sensors 23(13):5941;
# corroborated by multiple independent secondary sources): "Srate = rate of
# outbound packets' transmission in a flow", "Drate = rate of inbound
# packets' transmission in a flow" — directly equivalent to fwd_pkt_rate /
# bwd_pkt_rate (forward/backward packet rate), just computed over this
# extractor's sliding-window unit instead of a CICFlowMeter 5-tuple flow.
#
# Explicitly REJECTED: mapping the "Duration" column to "flow_duration".
# It looks tempting from the name alone, but the same paper's feature table
# documents "Duration" as the packet's IP Time-To-Live (TTL) — confirmed by
# its value range (0-255, matching an 8-bit TTL field) in the paper's
# descriptive statistics table. Mapping it to flow_duration would be a
# genuine semantic error (different physical quantity), not a coverage gain.
# This dataset's actual "flow_duration" column (present in the official
# 47-feature CICIoT2023 schema) is simply ABSENT from this 46-col CICIoMT2024
# re-upload — a genuine schema gap, not an alias-mapping oversight.
#
# Also explicitly NOT mapped (genuine schema mismatch, verified against the
# same source): Protocol Type (categorical, not a canonical feature),
# ece_flag_number / cwr_flag_number (no ECE/CWR slots in the 46-feature
# vocabulary), all protocol-presence one-hot flags (HTTP/HTTPS/DNS/.../LLC —
# no protocol-indicator slot exists in the vocabulary), and Magnitude /
# Radius / Covariance / Weight (composite cross-direction statistics with
# no single-feature canonical equivalent — e.g. Weight is explicitly defined
# as incoming_packets x outgoing_packets, a multiplicative composite that
# does not correspond to any additive canonical count we have).
# Coverage after this correction: 17/46 (37%) — up from 15/46, verified by
# re-running build_semantic_vector() against the real sample CSVs.
# NOTE: Short aliases (Min, Max, AVG, Std, IAT, Rate) are length < 5 chars.
# They are correctly handled by the exact case-insensitive match in
# build_semantic_vector() step 2, which runs BEFORE the substring guard.
# The substring guard in step 3 (len < 5 → skip) only applies to substring
# fallback — exact alias match is always attempted first and has no length guard.
# Coverage: 15/46 (genuine ceiling — extractor design limitation)

# ── ALIAS_UNSW ────────────────────────────────────────────────────────────
# UNSW-NB15 (Moustafa & Slay, MilCIS 2015). Argus + Bro/Zeek extractor — a
# THIRD, genuinely distinct extractor family (neither CICFlowMeter nor the
# CIC-IoT sliding-window statistical extractor). This is the heterogeneity
# upgrade: BRIDGE now spans three independent flow-extraction toolchains.
# Enterprise/general network testbed (NOT IoT-specific — same scope caveat as
# CICIDS-2017; see UNSW_SCOPE_NOTE below).
#
# Verified against the CLEAN labelled split shipped by the Kaggle mirror
# (mrwellsdavid/unsw-nb15): UNSW_NB15_training-set.csv / _testing-set.csv —
# 45 columns WITH a header, binary numeric `label` (0=normal, 1=attack) and a
# human-readable `attack_cat` family string. load_unsw_nb15() reads ONLY files
# that expose BOTH a `label` and a `dur` header column, which automatically
# rejects the raw headerless UNSW-NB15_1..4.csv captures and the
# NUSW-NB15_features.csv metadata file.
#
# Mappings follow the ORIGINAL 2015 feature definitions (Moustafa & Slay),
# mapped by PHYSICAL quantity, not by name:
#   dur    = record total duration                    -> flow_duration
#   spkts  = source->dest packet count                -> pkt_count_fwd
#   dpkts  = dest->source packet count                -> pkt_count_bwd
#   sbytes = source->dest transaction bytes           -> byte_count_fwd
#                                                        + fwd_pkt_len_total
#   dbytes = dest->source transaction bytes           -> byte_count_bwd
#                                                        + bwd_pkt_len_total
#   smean  = mean packet size, source direction       -> fwd_pkt_len_mean
#   dmean  = mean packet size, dest direction         -> bwd_pkt_len_mean
#   sinpkt = source inter-packet arrival time (mSec)  -> fwd_iat_mean
#   dinpkt = dest inter-packet arrival time (mSec)    -> bwd_iat_mean
#   sload  = source data rate (BITS/sec)              -> fwd_byte_rate  [A]
#   dload  = dest data rate (BITS/sec)                -> bwd_byte_rate  [A]
#   swin   = source TCP window advertisement value    -> init_win_fwd   [B]
#
# [A] sload/dload are BITS/sec; fwd_byte_rate/bwd_byte_rate are BYTES/sec — a
#     constant 8x unit factor fully absorbed by the per-feature RobustScaler
#     (scale-invariant), so the mapping is honest at the level of
#     "forward/backward throughput". The unit difference is noted, not hidden.
# [B] swin is the advertised TCP receive-window (source side); init_win_fwd
#     (CICFlowMeter) is the initial forward-window byte count. Both encode the
#     forward TCP window size; mapped with this caveat noted.
#
# EXPLICITLY REJECTED (kept unmapped on purpose — Section 3.4.4 discipline):
#   * rate -> pkt_rate : the derived `rate` column's exact physical definition
#     is NOT in Moustafa & Slay's canonical 49-feature table (added only in the
#     derived train/test split). Rather than guess it is the flow packet-rate,
#     it is left unmapped. (Loss: 1 slot.)
#   * sjit/djit -> iat_std : jitter (mean deviation of packet delay) and the
#     std of inter-arrival times are correlated but NOT the same statistic. No
#     canonical "jitter" slot exists; mapping to iat_std would be a semantic
#     approximation, so it is rejected. (Loss: 2 slots.)
#   * synack/ackdat -> flag_* : these are TCP setup *times* (SYN->SYN_ACK,
#     SYN_ACK->ACK), NOT flag counts. UNSW-NB15's clean split carries NO TCP
#     flag-count columns (only a categorical `state` and these setup times), so
#     ALL flag_* slots are correctly left unmapped — a genuine schema gap, not
#     an oversight. This differing coverage profile vs the CICFlowMeter datasets
#     is exactly the point of a heterogeneous benchmark.
#   * sttl/dttl (IP TTL), stcpb/dtcpb (TCP base seq), tcprtt, sloss/dloss, dwin,
#     proto/service/state (categorical), ct_* (connection-count aggregates),
#     trans_depth/response_body_len (HTTP), is_* : no canonical flow-statistic
#     equivalent — correctly zero-filled.
#
# Effective coverage: 14/46 canonical slots resolve (12 distinct source
# columns; sbytes/dbytes each fill two slots, exactly as ALIAS_FLOWMETER
# dual-maps byte_count_* and *_pkt_len_total). pkt_count_total is additionally
# derived in combine() from fwd+bwd (not read from a column), same as the other
# datasets. Confirm the live number via Cell 6's coverage table and the Cell 2b
# diagnostic after attaching the real file.
ALIAS_UNSW = {
    "flow_duration":     ["dur"],
    "pkt_count_fwd":     ["spkts"],
    "pkt_count_bwd":     ["dpkts"],
    "byte_count_fwd":    ["sbytes"],
    "byte_count_bwd":    ["dbytes"],
    "fwd_pkt_len_total": ["sbytes"],
    "bwd_pkt_len_total": ["dbytes"],
    "fwd_pkt_len_mean":  ["smean"],
    "bwd_pkt_len_mean":  ["dmean"],
    "fwd_iat_mean":      ["sinpkt"],
    "bwd_iat_mean":      ["dinpkt"],
    "fwd_byte_rate":     ["sload"],
    "bwd_byte_rate":     ["dload"],
    "init_win_fwd":      ["swin"],
}

ALIAS_MAPS = {
    "CICIDS-2017":                          ALIAS_FLOWMETER,
    "CIC-BCCC-NRC-TabularIoTAttacks-2024":  ALIAS_FLOWMETER,
    "CICIoMT-2024":                         ALIAS_IOMT,
    "UNSW-NB15":                             ALIAS_UNSW,
}

def build_semantic_vector(df, ds_name):
    alias_map = ALIAS_MAPS.get(ds_name, {})
    col_lower = {c.strip().lower(): c for c in df.columns}
    out = np.zeros((len(df), N_SEM), dtype=np.float32)
    matched = []
    for sem_name in SEMANTIC_FEATURES:
        si = SEM_IDX[sem_name]
        col = None
        # 1. Exact match (case-insensitive)
        if sem_name in col_lower:
            col = col_lower[sem_name]
        # 2. Alias match (exact, case-insensitive — no length guard here)
        elif sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower()
                if al in col_lower:
                    col = col_lower[al]; break
        # 3. Substring fallback — MINIMUM LENGTH GUARD (len < 5 → skip)
        # Short aliases like "Min", "Max", "Std", "IAT", "Rate" must NOT
        # match via substring — prevents false hits on unrelated columns.
        if col is None and sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower()
                if len(al) < 5:
                    continue
                ms = [c for c in df.columns if al in c.strip().lower()]
                if ms: col = ms[0]; break
        # 3b. Normalised substring fallback (strip spaces/underscores)
        if col is None and sem_name in alias_map:
            for alias in alias_map[sem_name]:
                al = alias.strip().lower().replace(' ','').replace('_','')
                if len(al) < 5:
                    continue
                for c in df.columns:
                    cn = c.strip().lower().replace(' ','').replace('_','')
                    if al == cn or (len(al) > 4 and al in cn):
                        col = c; break
                if col: break
        if col is not None:
            vals = pd.to_numeric(df[col], errors='coerce').fillna(0).values
            out[:, si] = vals.astype(np.float32)
            matched.append(sem_name)
    return out, matched

print(f"Alias maps defined for {len(ALIAS_MAPS)} datasets")
for ds, am in ALIAS_MAPS.items():
    print(f"  {ds}: {len(am)} alias entries / {N_SEM} canonical = "
          f"{len(am)/N_SEM*100:.0f}% (alias entries; actual coverage verified at runtime)")
print()
print("NOTE: actual coverage verified at runtime by build_semantic_vector().")
print("Run Cell 2b (diagnostic) after Cell 6 to confirm per-dataset coverage.")
print("Short aliases (<5 chars) only match via exact alias lookup, NOT substring.")

# ── ZERO-FILL SEMANTICS NOTE (paper Section 3.4.1) ───────────────────────────
# Absent features are zero-filled: conflates "true value is zero" with
# "feature not observed". This is a known limitation, disclosed in the paper.
# Mitigation: combine()._balance() enforces 1:1 class balance within each
# dataset block, removing the shortcut-learning risk. Cell 7b verifies this
# empirically. Post-scaling zero-fill displacement is bounded by clip(±10).
# CICIDS-2017 scope note: CICIDS-2017 is a general enterprise IDS benchmark
# (Sharafaldin et al., ICISSP 2018), not an IoT-specific botnet dataset.
# It is retained for canonical vocabulary coverage (91%) and cross-extractor
# domain shift testing. See paper Section 3.3.1 and Section 6 (Limitations).
ZERO_FILL_NOTE = (
    "Absent features zero-filled — conflation with true-zero is a known "
    "limitation. See Cell 7b shortcut test and paper Section 3.4.1."
)
UNSW_SCOPE_NOTE = (
    "UNSW-NB15 is a general enterprise/network testbed (Moustafa & Slay 2015), "
    "NOT IoT-specific. Retained as a THIRD extractor family (Argus/Bro) for "
    "cross-extractor generalisation — same scope caveat as CICIDS-2017."
)
CICIDS2017_SCOPE_NOTE = (
    "CICIDS-2017 is a general enterprise IDS benchmark, NOT IoT-specific. "
    "Retained as vocabulary anchor and cross-extractor shift test only."
)
print(f"Zero-fill policy: {ZERO_FILL_NOTE}")
print(f"CICIDS-2017 scope: {CICIDS2017_SCOPE_NOTE}")

In [ ]:
class Config:
    # ── Kaggle dataset paths ──────────────────────────────────────────────
    # PRIMARY (flow-level, all CICFlowMeter-compatible)
    CICIDS_PATH  = "/kaggle/input/datasets/bertvankeulen/cicids-2017"
    BCCC_PATH    = "/kaggle/input/datasets/kabeleswarpe/cic-bccc-nrc-tabulariotattacks-2024"
    IOMT_PATH    = "/kaggle/input/datasets/sinfdosh1990/ciciomt2024"
    UNSW_PATH    = "/kaggle/input/datasets/mrwellsdavid/unsw-nb15"

    # Auto-detect
    USE_CICIDS  = os.path.isdir(CICIDS_PATH) if CICIDS_PATH else False
    USE_BCCC    = os.path.isdir(BCCC_PATH)   if BCCC_PATH   else False
    USE_IOMT    = os.path.isdir(IOMT_PATH)   if IOMT_PATH   else False
    USE_UNSW    = os.path.isdir(UNSW_PATH)   if UNSW_PATH   else False

    # ── Row caps per dataset ──────────────────────────────────────────────
    CICIDS_MAX   = 500_000
    BCCC_MAX     = 3_000_000   # BCCC aggregates 9 IoT datasets, 1M+ records
    IOMT_MAX     = 3_000_000
    UNSW_MAX     = 500_000     # UNSW-NB15 train+test ~257k rows (cap not hit)

    # ── Sequence parameters ───────────────────────────────────────────────
    WINDOW_SIZE = 32
    STRIDE      = 4
    MAX_TRAIN_SEQ = 800_000
    MAX_TEST_SEQ  = 200_000
    MAX_VAL_SEQ   = 200_000

    # ── Class balance ─────────────────────────────────────────────────────
    TARGET_ATK_BEN_RATIO = 1.0   # strict 1:1 → ~43% attack windows after windowing

    # ── Training ──────────────────────────────────────────────────────────
    EPOCHS        = 30
    WARMUP        = 3
    EARLY_STOP    = 7
    BATCH_SIZE    = 512
    LR            = 5e-4
    WD            = 1e-4
    FOCAL_GAMMA   = 2.5
    LABEL_SMOOTH  = 0.01
    AUX_WT        = 0.05
    DEEP_SUP_WT   = 0.3     # per-expert CE weight (deep supervision; prevents gate collapse)
    CORAL_WT      = 0.5     # source-domain CORAL alignment weight (0 disables; ablatable)
    TTBN_ADAPT    = True    # transductive test-time BatchNorm adaptation at LODO eval

    # ── Architecture ──────────────────────────────────────────────────────
    EMBED_DIM     = 32
    CONV_CH       = [64, 128, 128]
    GRU_HIDDEN    = 128
    GRU_LAYERS    = 2
    ATTN_HEADS    = 8
    DROPOUT       = 0.20
    CBGAF_DIM     = 128

    # ── Evaluation ────────────────────────────────────────────────────────
    EVAL_SEEDS     = [42, 123, 456, 789, 2024]
    FAST_SEEDS     = [42, 123, 456]
    # REVISION (R1-M1): baselines use the SAME 5 seeds as TCH-Net so every
    # in-distribution paired test is matched seed-for-seed (n = 5 pairs).
    BASE_SEEDS     = [42, 123, 456, 789, 2024]
    LODO_SEEDS     = [42, 123, 456]   # 3 seeds per fold (descriptive spread only; no paired test)
    ABL_EPOCHS     = 30
    BASE_EPOCHS    = 30
    LODO_EPOCHS    = 30
    WARMUP_FAST    = 5

    N_DEV_CATS = 6   # 0=sensor, 1=camera, 2=appliance, 3=IIoT, 4=server, 5=unknown

    # ── Realistic deployment prior ─────────────────────────────────
    # Attack prevalence in real IoT traffic is typically 1-10%.
    # We evaluate at 5% to simulate realistic deployment conditions.
    REALISTIC_ATK_PRIOR = 0.05   # P(attack) in realistic traffic

    # ── HP sensitivity ranges ──────────────────
    HP_SENS_CONFIGS = [
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.20},   # default
        {'lr': 1e-3, 'wd': 1e-4, 'dropout': 0.20},   # 2x LR
        {'lr': 2e-4, 'wd': 1e-4, 'dropout': 0.20},   # 0.4x LR
        {'lr': 5e-4, 'wd': 1e-3, 'dropout': 0.20},   # 10x WD
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.30},   # higher dropout
        {'lr': 5e-4, 'wd': 1e-4, 'dropout': 0.10},   # lower dropout
    ]
    HP_SENS_SEEDS = [42, 123]   # 2 seeds × 6 configs = 12 runs
    N_CLASSES  = 2   # benign vs attack (binary IDS)
    N_DS_SRC   = 4   # CICIDS-2017=0, BCCC=1, CICIoMT-2024=2, UNSW-NB15=3

    # Device category map: dataset → {device_id: category_id}
    # device_id is an arbitrary per-loader integer; category is one of N_DEV_CATS
    DEVICE_CAT_MAP = {
        "CICIDS-2017":                          {0: 4},   # 0→server/workstation
        "CIC-BCCC-NRC-TabularIoTAttacks-2024":  {1: 0},   # 1→IoT sensor/device (mixed: cameras, lights, sensors)
        "CICIoMT-2024":                         {2: 3},   # 2→IIoT/medical device
        "UNSW-NB15":                            {3: 4},   # 3→server/workstation (enterprise, non-IoT)
    }

    # ── Reviewer-requested fixes ──────────────────────────────────────────────
    # chronological split is the DEFAULT.
    SPLIT_MODE      = 'chronological'   # 'chronological' | 'random'

    # REVISION (R1-m1): explicit validation subset for early stopping and
    # checkpoint selection. Carved chronologically per dataset BEFORE windowing:
    # train | val | test = 70 | 10 | 20 % of rows in row order. The test rows are
    # identical to the previous two-way split and are never used for selection.
    VAL_FRAC        = 0.10
    # LODO: validation is carved the same way from the SOURCE datasets only —
    # the held-out target domain is never used for early stopping.
    LODO_VAL_FRAC   = 0.10

    # feature-availability mask
    USE_MISS_MASK   = True

    # common seed set for all paired tests (REVISION R1-M1: 5 matched seeds)
    COMMON_SEEDS    = [42, 123, 456, 789, 2024]
    STAT_ALPHA      = 0.05   # family-wise level for Bonferroni-corrected paired tests

    # IDS operational metrics
    FIXED_FPR_LEVELS   = [0.001, 0.01, 0.05]
    FIXED_TPR_LEVELS   = [0.95, 0.99]  # also computes FPR@TPR99 for every model

    OUT = "tch_net_v2_results"

    # ── LODO baseline names (5 DL models run in LODO alongside TCH-Net) ────
    # Transformer-IDS, BiGRU-IDS, MLP-IDS, CNN-LSTM, 1D-CNN-IDS
    LODO_BASELINE_NAMES = [
        "Transformer-IDS",
        "BiGRU-IDS",
        "MLP-IDS",
        "CNN-LSTM",
        "1D-CNN-IDS",
    ]

os.makedirs(Config.OUT, exist_ok=True)
PAL = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
       '#00BCD4','#795548','#E91E63','#607D8B','#CDDC39']
def SAVE(n, fig=None):
    """Save current figure and display inline in Jupyter."""
    _f = fig if fig is not None else plt.gcf()
    _f.savefig(os.path.join(Config.OUT, n), bbox_inches='tight', dpi=150)

print("Dataset availability:")
for n, f in [("CICIDS-2017", Config.USE_CICIDS),
             ("CIC-BCCC-NRC-TabularIoTAttacks-2024", Config.USE_BCCC),
             ("CICIoMT-2024", Config.USE_IOMT),
             ("UNSW-NB15", Config.USE_UNSW)]:
    print(f"  {n:<38}: {'FOUND' if f else 'NOT FOUND'} [PRIMARY]")


In [ ]:
ATTACK_TYPE_STORE = {}

def _subsample(X, y, max_n, extras=None, seed=42):
    """
    Subsample to at most max_n rows, preserving the original class ratio
    but enforcing a minimum of 5000 samples per class when possible.
    The actual class re-balancing to 1:1 happens later in combine()._balance().
    This function only caps extremely large datasets to avoid OOM.
    """
    if max_n is None or len(X) <= max_n:
        return (X, y) + (tuple(extras) if extras else ())
    rng = np.random.RandomState(seed)
    ben = np.where(y==0)[0]; atk = np.where(y==1)[0]
    r = len(ben) / len(y)
    nb = min(len(ben), max(5_000, int(max_n * r)))
    na = min(len(atk), max(5_000, max_n - nb))
    if nb + na > max_n:
        scale = max_n / (nb + na)
        nb = max(5_000, int(nb * scale))
        na = max(5_000, min(max_n - nb, int(na * scale)))
    nb = min(nb, len(ben)); na = min(na, len(atk))
    keep = np.concatenate([rng.choice(ben, nb, replace=False),
                           rng.choice(atk, na, replace=False)])
    rng.shuffle(keep)
    result = [X[keep], y[keep]]
    if extras:
        for a in extras: result.append(a[keep])
    print(f"  Subsampled {len(X):,} -> {len(keep):,} "
          f"(ben={nb:,} atk={na:,} atk%={na/(nb+na)*100:.1f}%)")
    return tuple(result)

def augment_minority(X, y, target=5000, noise_std=0.01, seed=42):
    rng = np.random.RandomState(seed)
    ben_idx = np.where(y == 0)[0]
    atk_idx = np.where(y == 1)[0]
    minority_idx = ben_idx if len(ben_idx) < len(atk_idx) else atk_idx
    minority_label = 0 if len(ben_idx) < len(atk_idx) else 1
    n_minority = len(minority_idx)
    if n_minority >= target:
        return X, y
    n_needed = target - n_minority
    print(f"    Augmenting minority ({n_minority} -> {target}, "
          f"generating {n_needed} synthetic rows via Gaussian noise sigma={noise_std})")
    chosen = rng.choice(minority_idx, n_needed, replace=True)
    X_aug = X[chosen] + rng.normal(0, noise_std,
                                    size=(n_needed, X.shape[1])).astype(np.float32)
    X_aug = np.clip(X_aug, X.min(axis=0), X.max(axis=0))
    y_aug = np.full(n_needed, minority_label, dtype=np.int32)
    X_new = np.vstack([X, X_aug])
    y_new = np.hstack([y, y_aug])
    print(f"    After augmentation: ben={(y_new==0).sum():,} atk={(y_new==1).sum():,}")
    return X_new, y_new

def _is_benign_mask(lbl_series):
    """
    Robust benign/attack split for an object-dtype label column.
    Matches an exact small set of known benign tokens AND any label that
    merely CONTAINS "benign" (case-insensitive) — covers "Benign",
    "BENIGN", "BenignTraffic", "Benign Traffic", "Benign_Traffic", etc.
    without having to enumerate every uploader's exact string.
    """
    s = lbl_series.astype(str).str.strip().str.lower()
    exact = s.isin(['benign', 'benigntraffic', 'normal', '0'])
    contains = s.str.contains('benign', regex=False)
    return exact | contains

# ── CICIDS-2017 ──────────────────────────────────────────────────────────
# Verified against: cicids-2017-friday.csv, cicids-2017-friday-plus.csv
# (89 / 105 columns — "-plus" adds extra Flow ID / "Local_*" metadata columns
# that are not part of the canonical feature set and are safely ignored).
# Label column: "Label" (string, e.g. "BENIGN"). Uses ALIAS_FLOWMETER.
def load_cicids2017(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CICIDS-2017\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True) +
                   glob.glob(os.path.join(data_dir,"**","*.parquet"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    for fp in files:
        print(f"  Loading {os.path.basename(fp)}...")
        try:
            df = pd.read_parquet(fp) if fp.endswith('.parquet') else pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            lc = next((c for c in df.columns if c.lower().strip() == 'label'), None)
            if lc is None: print(f"    No label column"); continue
            y = (~_is_benign_mask(df[lc])).values.astype(np.int32)
            atk = np.where(y==0, 'BENIGN', df[lc].astype(str).str.strip().values)
            X_sem, matched = build_semantic_vector(df, "CICIDS-2017")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            print(f"    {len(y):,} rows | coverage={len(matched)}/{N_SEM} | "
                  f"Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,}")
        except Exception as e: print(f"    Skip: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 0 → server/workstation category (Config.DEVICE_CAT_MAP["CICIDS-2017"])
    dev = np.zeros(len(y), dtype=np.int32)
    ATTACK_TYPE_STORE['CICIDS-2017'] = atk
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,}")
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

# ── CIC-BCCC-NRC-TabularIoTAttacks-2024 ──────────────────────────────────
# Replaces the broken CIC-IoT-DIAD-2024 (Mostafa Ali Ogab) — that dataset's
# Label column was "NeedManualLabel" on EVERY row, including attack-named
# files. Confirmed unusable; see Cell 2 dataset-reset note.
#
# BCCC research center (York University). CICFlowMeter-extracted, 85
# columns, aggregates 9 IoT datasets into one unified table. 1M+ records.
# Verified against: cic-bccc-nrc-tabulariotAttack.csv
#   Label column: "Label" — int, 0 in the (benign-only) sample.
#   "Attack Name" column: string, e.g. "Benign Traffic" in the sample.
# Label strategy: prefer the explicit "Attack Name" string (unambiguous
# human-readable family) when present; fall back to numeric Label
# (0=benign / nonzero=attack — works whether Label is strictly binary or a
# multi-class integer encoding, since != 0 is correct in both cases).
# Kaggle: kabeleswarpe/cic-bccc-nrc-tabulariotattacks-2024
def load_cic_bccc_nrc(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CIC-BCCC-NRC-TabularIoTAttacks-2024\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','summary','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    _agree_n = 0; _agree_match = 0   # label cross-validation tally (see below)
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True

            attack_col = next((c for c in df.columns if c.lower().strip() == 'attack name'), None)
            label_col  = next((c for c in df.columns if c.lower().strip() == 'label'), None)

            # ── Label cross-validation ──────────────────────────────────────
            # This dataset carries TWO independent label signals per row:
            # a human-readable family string ("Attack Name") and a numeric
            # code ("Label"). Rather than trusting one and discarding the
            # other, derive y from BOTH whenever both are present and report
            # their agreement rate. Attack Name is used as the actual ground
            # truth (explicit, unambiguous string) — Label is a corroborating
            # signal, not a silent fallback. Any disagreement is surfaced
            # immediately instead of being silently absorbed into the labels.
            if attack_col is not None and label_col is not None:
                atk_str = df[attack_col].astype(str).str.strip()
                y_from_attack = (~atk_str.str.lower().str.contains('benign')).values.astype(np.int32)
                lbl = df[label_col]
                if lbl.dtype == object:
                    y_from_label = (~_is_benign_mask(lbl)).values.astype(np.int32)
                else:
                    y_from_label = (lbl.values != 0).astype(np.int32)
                _agree = (y_from_attack == y_from_label)
                _agree_n += len(_agree); _agree_match += int(_agree.sum())
                if not _agree.all():
                    print(f"    NOTE: {int((~_agree).sum())}/{len(_agree)} rows in "
                          f"{os.path.basename(fp)} disagree between 'Attack Name' "
                          f"and numeric 'Label' — using 'Attack Name'")
                y = y_from_attack
                atk = atk_str.values
            elif attack_col is not None:
                atk_str = df[attack_col].astype(str).str.strip()
                y = (~atk_str.str.lower().str.contains('benign')).values.astype(np.int32)
                atk = atk_str.values
            elif label_col is not None:
                lbl = df[label_col]
                if lbl.dtype == object:
                    y = (~_is_benign_mask(lbl)).values.astype(np.int32)
                    atk = lbl.astype(str).values
                else:
                    y = (lbl.values != 0).astype(np.int32)
                    atk = np.where(y==0, 'Benign Traffic', lbl.astype(str).values)
            else:
                print(f"  No 'Attack Name' or 'Label' column in {os.path.basename(fp)} — skip")
                continue

            X_sem, matched = build_semantic_vector(df, "CIC-BCCC-NRC-TabularIoTAttacks-2024")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
            if max_samples is not None and _acc_rows >= max_samples * 2:
                print(f"  Early exit at {_acc_rows:,} rows (max_samples={max_samples:,})")
                break
        except Exception as e: print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    if _agree_n > 0:
        _agree_pct = _agree_match / _agree_n * 100
        print(f"\n  Label cross-validation ('Attack Name' vs numeric 'Label'): "
              f"{_agree_match:,}/{_agree_n:,} rows agree ({_agree_pct:.2f}%)")
        if _agree_pct < 99.0:
            print(f"  WARNING: >1% disagreement between the two label columns — "
                  f"inspect before trusting either derivation in the paper.")
    # device_id 1 → generic IoT device category (BCCC mixes cameras/lights/sensors)
    dev = np.ones(len(y), dtype=np.int32)
    ATTACK_TYPE_STORE['CIC-BCCC-NRC-TabularIoTAttacks-2024'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5:
        print(f"  WARNING: very low attack% — check label parsing")
    rng_bccc = np.random.RandomState(42)
    perm = rng_bccc.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

# ── CICIoMT-2024 (Train-Test, D. Kilichev) ───────────────────────────────
# Replaces the broken CICIoMT-2024 (Lian Mollick) variant — that upload had
# 45 columns and NO label column at all (same reduced schema as CIC-IoT-2023).
# Confirmed unusable; see Cell 2 dataset-reset note.
#
# Lashkari et al. (2024). Custom CIC statistical extractor (46-col schema
# incl. Label — identical feature schema to the broken variant, but THIS
# upload's Label column is genuinely populated).
# Verified against: ciciomt-train-set.csv ("DoS-TCP"), ciciomt-test-set.csv
# ("Malformed_Data"). Neither 10-row sample contains a benign row, so the
# exact benign string token is unconfirmed — _is_benign_mask() matches any
# label containing "benign" case-insensitively to stay robust to whichever
# exact spelling ("Benign" / "BenignTraffic" / "Benign Traffic") this
# upload uses.
# Label column: "Label". 17/46 features alias-resolve; 16/46 (35%) carry
# non-trivial signal on the full live dataset (Drate/bwd_pkt_rate resolves
# correctly but is ~0 across virtually every real row — matches the source
# paper's own published stats, not a mapping defect). See Cell 2 for the
# full Srate/Drate/Duration reasoning.
def load_ciciomt(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING CICIoMT-2024\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','summary','description'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True
            lc = next((c for c in df.columns if c.lower().strip() == 'label'), None)
            if lc is None:
                # Some file splits carry no label — skip silently
                continue
            lbl = df[lc]
            if lbl.dtype == object:
                y = (~_is_benign_mask(lbl)).values.astype(np.int32)
                atk = lbl.astype(str).values
            else:
                y = (lbl.values != 0).astype(np.int32)
                atk = np.where(y==0, 'Benign', lbl.astype(str).values)
            X_sem, matched = build_semantic_vector(df, "CICIoMT-2024")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
            if max_samples is not None and _acc_rows >= max_samples * 2:
                print(f"  Early exit at {_acc_rows:,} rows (max_samples={max_samples:,})")
                break
        except Exception as e: print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 2 → IIoT/medical device category
    dev = np.full(len(y), 2, dtype=np.int32)
    ATTACK_TYPE_STORE['CICIoMT-2024'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5:
        print(f"  WARNING: very low attack% — check label parsing")
    rng_iomt = np.random.RandomState(42)
    perm = rng_iomt.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

# \u2500\u2500 UNSW-NB15 (Moustafa & Slay 2015; Argus + Bro/Zeek) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# THIRD extractor family \u2014 the heterogeneity upgrade. Enterprise/general
# network testbed (NOT IoT-specific \u2014 same scope caveat as CICIDS-2017).
# Reads ONLY the clean labelled split (files exposing BOTH a `label` and a
# `dur` header column): UNSW_NB15_training-set.csv / _testing-set.csv.
# This automatically SKIPS:
#   * the raw headerless captures UNSW-NB15_1..4.csv (no header -> first data
#     row misread as header -> lacks `label`/`dur` -> rejected), and
#   * NUSW-NB15_features.csv / *LIST_EVENTS* metadata files.
# Label: numeric `label` (0=normal, 1=attack). `attack_cat` gives the family.
# Uses ALIAS_UNSW. device_id 3 -> enterprise/server category (DEVICE_CAT_MAP).
# Kaggle: mrwellsdavid/unsw-nb15
def load_unsw_nb15(data_dir, max_samples=None):
    print("\n"+"="*70+"\nLOADING UNSW-NB15\n"+"="*70)
    files = sorted(glob.glob(os.path.join(data_dir,"**","*.csv"), recursive=True))
    files = [f for f in files if not any(kw in os.path.basename(f).lower()
             for kw in ['readme','feature','description','list_events',
                        'list-events','ground_truth','_gt'])]
    if not files: print(f"  No files in {data_dir}"); return None, None, None

    X_all, y_all, atk_all = [], [], []
    _printed_cols = False
    _acc_rows = 0
    for fp in files:
        try:
            df = pd.read_csv(fp, low_memory=False)
            df.columns = df.columns.str.strip()
            cols_lower = {c.lower() for c in df.columns}
            # Guard: accept ONLY the clean labelled split. Raw headerless
            # captures and the features-metadata file lack this exact pair.
            if 'label' not in cols_lower or 'dur' not in cols_lower:
                print(f"  Skip {os.path.basename(fp)}: not the clean labelled "
                      f"schema (needs 'label'+'dur' header) \u2014 raw/meta file")
                continue
            if not _printed_cols:
                print(f"  Columns ({len(df.columns)}): {df.columns.tolist()[:20]}")
                if len(df.columns) > 20: print(f"    ... and {len(df.columns)-20} more")
                _printed_cols = True
            lc = next(c for c in df.columns if c.lower().strip() == 'label')
            lbl = df[lc]
            if lbl.dtype == object:
                y = (~_is_benign_mask(lbl)).values.astype(np.int32)
            else:
                y = (lbl.values != 0).astype(np.int32)   # 0=normal, 1=attack
            ac = next((c for c in df.columns if c.lower().strip() == 'attack_cat'), None)
            if ac is not None:
                atk = np.where(y==0, 'Normal', df[ac].astype(str).str.strip().values)
            else:
                atk = np.where(y==0, 'Normal', 'Attack')
            X_sem, matched = build_semantic_vector(df, "UNSW-NB15")
            np.nan_to_num(X_sem, copy=False, nan=0., posinf=1e9, neginf=-1e9)
            X_all.append(X_sem); y_all.append(y); atk_all.append(atk)
            _acc_rows += len(y)
            cov = len(matched)
            print(f"  {os.path.basename(fp)}: {len(y):,} rows | cov={cov}/{N_SEM} | "
                  f"Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} | acc={_acc_rows:,}")
            if cov == 0:
                print(f"    WARNING: 0 features matched! Actual columns: {list(df.columns[:15])}")
        except Exception as e:
            print(f"  Skip {os.path.basename(fp)}: {e}")

    if not X_all: return None, None, None
    X = np.vstack(X_all); y = np.hstack(y_all); atk = np.hstack(atk_all)
    # device_id 3 -> enterprise/server category (non-IoT, like CICIDS-2017)
    dev = np.full(len(y), 3, dtype=np.int32)
    ATTACK_TYPE_STORE['UNSW-NB15'] = atk
    atk_pct = float((y==1).mean()*100)
    print(f"  Total: {len(y):,} | Ben={int((y==0).sum()):,} Atk={int((y==1).sum()):,} "
          f"({atk_pct:.1f}% attack)")
    if atk_pct < 5 or atk_pct > 95:
        print(f"  WARNING: extreme attack% ({atk_pct:.1f}%) \u2014 check label parsing")
    rng_unsw = np.random.RandomState(42)
    perm = rng_unsw.permutation(len(X)); X, y, dev = X[perm], y[perm], dev[perm]
    X, y, dev = _subsample(X, y, max_samples, [dev])
    return X, y, dev

print("All 4 dataset loaders defined")


In [ ]:
class MultiDatasetLoader:
    # Warn if combined post-balance count < 2*MIN_FLOOR (signals scarce dataset).
    # Any dataset whose minority class is small will be flagged but still included.
    MIN_FLOOR = 5_000  # advisory threshold only

    def __init__(self):
        self.datasets = {}
        self.scaler = None
        # NOTE: X_train_scaled / X_test_scaled (stored after scaling)
        # Named *_raw to match usage in HP sensitivity cells.
        self.X_train_raw = self.X_test_raw = self.X_val_raw = None

    def add(self, name, X, y, dev, ds_src_id):
        if X is None: print(f"  Skip {name}: no data"); return
        cat_map = Config.DEVICE_CAT_MAP.get(name, {})
        dev_cats = np.array([cat_map.get(int(d), 5) for d in dev], dtype=np.int32)
        ds_ids   = np.full(len(y), ds_src_id, dtype=np.int32)
        ctx      = np.stack([ds_ids, dev_cats], axis=1)
        n_matched = int((X != 0).any(axis=0).sum())
        self.datasets[name] = {'X': X, 'y': y, 'dev': dev, 'ctx': ctx,
                               'ds_src': ds_src_id, 'n': X.shape[0]}
        print(f"  Added {name}: {X.shape[0]:,}x{X.shape[1]} "
              f"(non-zero dims: {n_matched}/{N_SEM} = {n_matched/N_SEM*100:.0f}%)")

    def _balance(self, X, y, ctx, ratio, rng, name=""):
        """
        Strict class balance with seeded RNG for reproducibility.

        ratio=1.0 → 50% attack rows → p(window=attack) ≈ 43% with majority-vote(w=32).

        MIN_FLOOR: advisory only — warns when a dataset contributes fewer than
        2*MIN_FLOOR total samples after balancing (poor signal, still included).
        The actual cap is ALWAYS strict: majority capped to ratio * minority.
        We never inflate the majority beyond ratio*minority just because minority
        is small — that would produce up to 5:1 imbalance and near-100% attack
        windows after windowing (empirically verified on prior, since-removed
        low-sample datasets in this pipeline's history).
        FocalLoss alpha weighting handles residual class imbalance at batch level.
        """
        ben = np.where(y == 0)[0]
        atk = np.where(y == 1)[0]
        if len(ben) == 0 or len(atk) == 0:
            print(f"  WARNING {name}: single-class data "
                  f"(ben={len(ben)}, atk={len(atk)}) — skipping balance")
            return X, y, ctx

        # Strict 1:1 balance always — even if minority class is tiny.
        # Reason: majority-vote windowing needs ~50% row balance to produce
        # mixed-label windows. A 20:1 imbalance produces ~100% attack windows,
        # giving the model zero benign examples from that dataset.
        # A dataset with very few minority-class rows is accepted as-is rather
        # than upsampled past this floor — FocalLoss alpha handles the residual.
        # FocalLoss alpha and the global _cap handle residual imbalance.
        MIN_SAMPLES = 5000
        if len(atk) > ratio * len(ben):
            if len(ben) >= MIN_SAMPLES:
                n_atk = int(ratio * len(ben))
                atk = rng.choice(atk, n_atk, replace=False)
            else:
                n_atk = min(len(atk), MIN_SAMPLES)
                atk = rng.choice(atk, n_atk, replace=False)
        elif len(ben) > ratio * len(atk):
            if len(atk) >= MIN_SAMPLES:
                n_ben = int(ratio * len(atk))
                ben = rng.choice(ben, n_ben, replace=False)
            else:
                n_ben = min(len(ben), MIN_SAMPLES)
                ben = rng.choice(ben, n_ben, replace=False)

        n_total = len(ben) + len(atk)
        if n_total < 2 * self.MIN_FLOOR:
            print(f"  ADVISORY {name}: only {n_total:,} samples after balancing "
                  f"(minority class was very small: {min(len(ben),len(atk)):,}). "
                  f"FocalLoss alpha weighting compensates during training.")

        keep = np.concatenate([ben, atk])
        # CRITICAL FIX: DO NOT shuffle here.
        # combine() performs a chronological split by sorting positional indices
        # within each dataset block. Shuffling here destroys temporal order
        # within the block, making ds_idx_sorted meaningless as a time-order
        # proxy. Instead, sort keep so rows are in their original temporal order.
        # _cap() shuffles windows after sequencing — randomness is preserved there.
        keep = np.sort(keep)
        return X[keep], y[keep], ctx[keep]

    def combine(self, ratio=None, test_size=0.2, seed=42):
        if ratio is None: ratio = Config.TARGET_ATK_BEN_RATIO
        rng = np.random.RandomState(seed)
        print("\n" + "="*70 + "\nCOMBINE & PREPROCESS\n" + "="*70)
        Xa, ya, ca, da = [], [], [], []

        for di, (name, d) in enumerate(self.datasets.items()):
            X_r = d['X'].copy(); y_r = d['y'].copy(); c_r = d['ctx'].copy()
            np.nan_to_num(X_r, copy=False, nan=0., posinf=1e6, neginf=-1e6)

            n_nonzero = int((X_r != 0).any(axis=0).sum())
            if n_nonzero == 0:
                print(f"  SKIP {name}: 0% feature coverage — all features zero.")
                continue
            if n_nonzero < 5:
                print(f"  WARNING {name}: very low coverage ({n_nonzero}/46). Results unreliable.")

            # ── FIX: derive pkt_count_total BEFORE scaling ───────────────────
            # Cannot do fwd_scaled + bwd_scaled ≠ scale(fwd+bwd). Must use raw counts.
            _fwd = SEM_IDX["pkt_count_fwd"]
            _bwd = SEM_IDX["pkt_count_bwd"]
            _tot = SEM_IDX["pkt_count_total"]
            zero_tot = (X_r[:, _tot] == 0)
            X_r[zero_tot, _tot] = X_r[zero_tot, _fwd] + X_r[zero_tot, _bwd]

            X_r, y_r, c_r = self._balance(X_r, y_r, c_r, ratio, rng, name)
            ben_r = int((y_r==0).sum()); atk_r = int((y_r==1).sum())
            est_seq = max(0, (len(y_r) - Config.WINDOW_SIZE)) // Config.STRIDE + 1
            print(f"  {name}: {len(y_r):,} rows "
                  f"(ben={ben_r:,} atk={atk_r:,} "
                  f"ratio={atk_r/max(ben_r,1):.1f}:1) "
                  f"est_sequences~{est_seq:,}")
            Xa.append(X_r); ya.append(y_r); ca.append(c_r)
            da.append(np.full(len(y_r), di, dtype=np.int32))

        if not Xa:
            raise RuntimeError("No datasets survived combine() — check feature coverage")

        X_c   = np.vstack(Xa).astype(np.float32)
        y_c   = np.hstack(ya).astype(np.int32)
        ctx_c = np.vstack(ca).astype(np.int32)
        ds_c  = np.hstack(da).astype(np.int32)

        total_ben = int((y_c == 0).sum()); total_atk = int((y_c == 1).sum())
        print(f"\n  Pre-split combined: {len(y_c):,} rows "
              f"| ben={total_ben:,} ({total_ben/len(y_c)*100:.1f}%) "
              f"| atk={total_atk:,} ({total_atk/len(y_c)*100:.1f}%)")

        # ── CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT (default) ──────────
        # REVISION (R1-m1). Split BEFORE window generation, per dataset, by
        # original row order:
        #   first (1 − test_size − VAL_FRAC) of rows → train
        #   next   VAL_FRAC                         → validation (early stopping
        #                                             and checkpoint selection ONLY)
        #   last   test_size                        → test (same rows as the
        #                                             previous two-way split)
        # Windows are generated separately inside each partition and never across
        # datasets, so no sliding window can contain rows from two partitions:
        # overlapping-window leakage between train, validation and test is
        # impossible by construction. Previously there was no validation subset
        # and early stopping monitored the test set.
        _vf = getattr(Config, 'VAL_FRAC', 0.10)
        if Config.SPLIT_MODE == 'chronological':
            i_tr, i_va, i_te = [], [], []
            for ds_id in np.unique(ds_c):
                # _balance() keeps rows in original (temporal) order and datasets
                # are concatenated block-wise, so sorted positions = time order.
                ds_idx_sorted = np.sort(np.where(ds_c == ds_id)[0])
                n_trva = int(len(ds_idx_sorted) * (1 - test_size))
                n_va   = int(len(ds_idx_sorted) * _vf)
                i_tr.append(ds_idx_sorted[:n_trva - n_va])
                i_va.append(ds_idx_sorted[n_trva - n_va:n_trva])
                i_te.append(ds_idx_sorted[n_trva:])
            i_tr, i_va, i_te = np.concatenate(i_tr), np.concatenate(i_va), np.concatenate(i_te)
            print(f"  Split mode: CHRONOLOGICAL (per-dataset train/val/test = "
                  f"{(1-test_size-_vf)*100:.0f}/{_vf*100:.0f}/{test_size*100:.0f}% by row order)")
        else:
            # Random split retained for ablation comparisons only
            _idx_all = np.arange(len(y_c))
            _idx_trva, i_te = train_test_split(_idx_all, test_size=test_size,
                                               stratify=y_c, random_state=seed)
            i_tr, i_va = train_test_split(_idx_trva, test_size=_vf / (1 - test_size),
                                          stratify=y_c[_idx_trva], random_state=seed)
            print(f"  Split mode: RANDOM stratified (ablation only — not default)")
        X_tr, y_tr, c_tr, d_tr = X_c[i_tr], y_c[i_tr], ctx_c[i_tr], ds_c[i_tr]
        X_va, y_va, c_va, d_va = X_c[i_va], y_c[i_va], ctx_c[i_va], ds_c[i_va]
        X_te, y_te, c_te, d_te = X_c[i_te], y_c[i_te], ctx_c[i_te], ds_c[i_te]

        # Scale AFTER pkt_count_total derivation (now done above, pre-split).
        # The scaler is fit on TRAIN rows only; validation and test are transformed.
        self.scaler = RobustScaler(quantile_range=(5, 95))
        X_tr = np.clip(self.scaler.fit_transform(X_tr), -10, 10).astype(np.float32)
        X_va = np.clip(self.scaler.transform(X_va),     -10, 10).astype(np.float32)
        X_te = np.clip(self.scaler.transform(X_te),     -10, 10).astype(np.float32)

        # Validate post-split balance
        for tag_, y_ in [('Train', y_tr), ('Val', y_va), ('Test', y_te)]:
            b_ = int((y_==0).sum()); a_ = int((y_==1).sum())
            print(f"  {tag_+':':<6} {len(y_):,} | ben={b_:,} ({b_/len(y_)*100:.1f}%) "
                  f"| atk={a_:,} ({a_/len(y_)*100:.1f}%)")
        # Per-dataset validation composition (disclosed in the paper's split description)
        _names_ = list(self.datasets.keys())
        for ds_id in np.unique(d_va):
            _m_ = (d_va == ds_id)
            _nm_ = _names_[ds_id] if ds_id < len(_names_) else f'DS{ds_id}'
            print(f"    val rows {_nm_:<38}: {int(_m_.sum()):,} "
                  f"(attack {float(y_va[_m_].mean())*100:.1f}%)")

        # Store scaled data for HP sensitivity reuse
        self.X_train_raw = X_tr; self.y_train_raw = y_tr
        self.ctx_train_raw = c_tr; self.ds_train_raw = d_tr
        self.X_val_raw   = X_va; self.y_val_raw   = y_va
        self.ctx_val_raw = c_va; self.ds_val_raw = d_va
        self.X_test_raw  = X_te; self.y_test_raw  = y_te
        self.ctx_test_raw = c_te; self.ds_test_raw = d_te

        print(f"  Scaler: single RobustScaler(q5,q95) fit on train only")
        # Post-split balance check
        for tag_, y_ in [('Train', y_tr), ('Test', y_te)]:
            r_ = float((y_==1).mean())
            if r_ < 0.05 or r_ > 0.95:
                raise RuntimeError(f'{tag_} set is severely imbalanced: attack%={r_*100:.1f}%. Check _balance().')
        _rv = float((y_va==1).mean())
        if _rv < 0.05 or _rv > 0.95:
            print(f"  WARNING: validation set is imbalanced (attack%={_rv*100:.1f}%); "
                  f"early stopping falls back to validation loss if it is single-class.")
        return X_tr, y_tr, c_tr, d_tr, X_te, y_te, c_te, d_te

loader = MultiDatasetLoader()
DS_SRC = {
    "CICIDS-2017":                          0,
    "CIC-BCCC-NRC-TabularIoTAttacks-2024":  1,
    "CICIoMT-2024":                         2,
    "UNSW-NB15":                             3,
}
print("MultiDatasetLoader ready")
print(f"Dataset source IDs: {DS_SRC}")


## A. Raw file order, timestamps and identifiers
Reads each dataset's files **in their original order, without shuffling**, and reports:
which timestamp / flow-ID / IP columns exist, whether rows are already time-ordered,
and how labels are grouped in file order (long label runs mean the file is sorted by class).
Answers Reviewer 3 §3.4 (can real chronological sequences be built?).

In [ ]:
AUDIT = {}
AUDIT_OUT = 'bridge_audit'; os.makedirs(AUDIT_OUT, exist_ok=True)

_TS_HINTS  = ('timestamp', 'stime', 'ltime', 'start time', 'time stamp', 'flow start')
_ID_HINTS  = ('flow id', 'flow_id', 'src ip', 'srcip', 'source ip', 'dst ip', 'dstip',
              'destination ip', 'src port', 'sport', 'dst port', 'dsport', 'destination port',
              'source port')

def _dataset_files(path, kind):
    fs = sorted(glob.glob(os.path.join(path, '**', '*.csv'), recursive=True))
    if kind == 'CICIDS-2017':
        fs += sorted(glob.glob(os.path.join(path, '**', '*.parquet'), recursive=True))
    bad = ['readme', 'feature', 'description', 'summary', 'list_events', 'list-events',
           'ground_truth', '_gt']
    return [f for f in fs if not any(k in os.path.basename(f).lower() for k in bad)]

def _read_cols(fp, usecols=None, nrows=None):
    if fp.endswith('.parquet'):
        df = pd.read_parquet(fp, columns=usecols)
        return df if nrows is None else df.head(nrows)
    return pd.read_csv(fp, low_memory=False, usecols=usecols, nrows=nrows,
                       skipinitialspace=True)

def _runs(y):
    """Number of label runs and mean run length in file order."""
    if len(y) == 0: return 0, 0.0
    n_runs = int(1 + np.count_nonzero(y[1:] != y[:-1]))
    return n_runs, float(len(y) / n_runs)

def _label_vec(df, kind):
    cols = {c.strip().lower(): c for c in df.columns}
    if kind == 'CIC-BCCC-NRC-TabularIoTAttacks-2024' and 'attack name' in cols:
        s = df[cols['attack name']].astype(str).str.strip().str.lower()
        return (~s.str.contains('benign')).values.astype(np.int8)
    if 'label' not in cols: return None
    lbl = df[cols['label']]
    if lbl.dtype == object: return (~_is_benign_mask(lbl)).values.astype(np.int8)
    return (lbl.values != 0).astype(np.int8)

_raw_paths = [('CICIDS-2017', Config.CICIDS_PATH, Config.USE_CICIDS),
              ('CIC-BCCC-NRC-TabularIoTAttacks-2024', Config.BCCC_PATH, Config.USE_BCCC),
              ('CICIoMT-2024', Config.IOMT_PATH, Config.USE_IOMT),
              ('UNSW-NB15', Config.UNSW_PATH, Config.USE_UNSW)]
AUDIT['raw_files'] = {}
for _name, _path, _use in _raw_paths:
    if not _use: continue
    print('\n' + '=' * 70 + f'\n{_name}\n' + '=' * 70)
    _rows = []
    for _fp in _dataset_files(_path, _name):
        try:
            _head = _read_cols(_fp, nrows=5)
        except Exception as e:
            print(f'  skip {os.path.basename(_fp)}: {e}'); continue
        _head.columns = [c.strip() for c in _head.columns]
        _cl = [c.lower() for c in _head.columns]
        if _name == 'UNSW-NB15' and not ('label' in _cl and 'dur' in _cl):
            continue
        _ts = [c for c in _head.columns if any(h in c.lower() for h in _TS_HINTS)]
        _ids = [c for c in _head.columns if c.lower() in _ID_HINTS]
        _lab = [c for c in _head.columns if c.lower() in ('label', 'attack name', 'attack_cat')]
        _sel = set(_ts + _ids + _lab) or {_head.columns[0]}
        try:
            if _fp.endswith('.parquet'):
                _df = pd.read_parquet(_fp)
                _df.columns = [c.strip() for c in _df.columns]
                _df = _df[[c for c in _df.columns if c in _sel]]
            else:
                _df = _read_cols(_fp, usecols=lambda c: c.strip() in _sel)
        except Exception as e:
            print(f'  skip {os.path.basename(_fp)}: {e}'); continue
        _df.columns = [c.strip() for c in _df.columns]
        _y = _label_vec(_df, _name)
        _info = {'file': os.path.basename(_fp), 'rows': int(len(_df)),
                 'n_columns': int(len(_head.columns)), 'timestamp_cols': _ts,
                 'id_cols': _ids}
        if _y is not None:
            _nr, _ml = _runs(_y)
            _info.update({'attack_frac': float(_y.mean()), 'label_runs': _nr,
                          'mean_label_run': round(_ml, 1)})
        for _t in _ts[:1]:
            _v = _df[_t]
            _num = pd.to_numeric(_v, errors='coerce')
            if _num.notna().mean() < 0.5:
                try:
                    _dt = pd.to_datetime(_v, errors='coerce', format='mixed')
                except (TypeError, ValueError):
                    _dt = pd.to_datetime(_v, errors='coerce')
                _num = (_dt - pd.Timestamp(0)).dt.total_seconds()
            _ok = _num.dropna().values
            if len(_ok) > 1:
                _d = np.diff(_ok.astype(np.float64))
                _info.update({'ts_col': _t, 'ts_parsed_frac': float(_num.notna().mean()),
                              'ts_nondecreasing_frac': float((_d >= 0).mean())})
        _rows.append(_info)
        print(f"  {_info['file'][:48]:<48} rows={_info['rows']:>9,} "
              f"ts={_ts[:2]} ids={_ids[:3]} "
              f"atk={_info.get('attack_frac', float('nan')):.2f} "
              f"runs={_info.get('label_runs', '-')} "
              f"monotone={_info.get('ts_nondecreasing_frac', float('nan')):.3f}")
        del _df; gc.collect()
    AUDIT['raw_files'][_name] = _rows
    if _rows:
        print(f"  first file columns: {list(_head.columns)}")

## B. Units of every mapped feature
For every canonical slot that a dataset fills, reports the raw source column, its
documented unit, and the 1st/50th/99th percentiles **before scaling**.
Consistency checks then test the documented units from the data itself, e.g.
for UNSW-NB15, `sload ≈ 8·sbytes/dur` holds only if `sload` is bits/s and `dur` seconds.
Answers Reviewer 3 §3.2 and Reviewer 4 comment 1.

In [ ]:
print('=' * 70 + '\nLOADING DATASETS (pipeline loaders, mapped canonical vectors)\n' + '=' * 70)
loader = MultiDatasetLoader()
DS_SRC = {'CICIDS-2017': 0, 'CIC-BCCC-NRC-TabularIoTAttacks-2024': 1,
          'CICIoMT-2024': 2, 'UNSW-NB15': 3}
for _name, _fn, _path, _use, _cap_n in [
        ('CICIDS-2017', load_cicids2017, Config.CICIDS_PATH, Config.USE_CICIDS, Config.CICIDS_MAX),
        ('CIC-BCCC-NRC-TabularIoTAttacks-2024', load_cic_bccc_nrc, Config.BCCC_PATH, Config.USE_BCCC, Config.BCCC_MAX),
        ('CICIoMT-2024', load_ciciomt, Config.IOMT_PATH, Config.USE_IOMT, Config.IOMT_MAX),
        ('UNSW-NB15', load_unsw_nb15, Config.UNSW_PATH, Config.USE_UNSW, Config.UNSW_MAX)]:
    if not _use: continue
    try:
        X, y, d = _fn(_path, _cap_n)
        if X is not None: loader.add(_name, X, y, d, DS_SRC[_name])
        del X, y, d
    except Exception as e:
        print(f'{_name} failed: {e}')
gc.collect()
DS_NAMES = list(loader.datasets.keys())

# Documented units (source papers / extractor documentation). 'us' = microseconds.
DOC_UNITS = {
    'CICIDS-2017': {'flow_duration': 'us', 'iat_*': 'us', '*_byte_rate|byte_rate': 'bytes/s',
                    '*_pkt_rate|pkt_rate': 'pkts/s', 'pkt_len_*|*_pkt_len_*': 'bytes',
                    'byte_count_*': 'bytes'},
    'CIC-BCCC-NRC-TabularIoTAttacks-2024': {'flow_duration': 'us (CICFlowMeter; verify)',
                    'iat_*': 'us (verify)', 'byte_rate': 'bytes/s', 'pkt_len_*': 'bytes'},
    'CICIoMT-2024': {'pkt_rate': 'pkts/s', 'iat_mean': 'undocumented (CICIoT2023 "IAT")',
                     'pkt_len_*': 'bytes', 'byte_count_total': 'bytes'},
    'UNSW-NB15': {'flow_duration': 's', 'fwd_iat_mean|bwd_iat_mean': 'ms',
                  'fwd_byte_rate|bwd_byte_rate': 'bits/s', 'pkt_len_mean|byte_count': 'bytes'},
}
AUDIT['doc_units'] = DOC_UNITS

AUDIT['units'] = {}
for _name, _d in loader.datasets.items():
    _X = _d['X']; _am = ALIAS_MAPS.get(_name, {})
    print('\n' + '-' * 70 + f'\n{_name}  ({_X.shape[0]:,} rows)\n' + '-' * 70)
    print(f"  {'canonical':<18} {'source column':<22} {'p01':>12} {'p50':>12} {'p99':>12} {'zero%':>6}")
    _rows = {}
    for _j, _f in enumerate(SEMANTIC_FEATURES):
        _v = _X[:, _j]
        if not (_v != 0).any(): continue
        _src = _am.get(_f, [_f])[0] if _am else _f
        _q = np.percentile(_v, [1, 50, 99])
        _rows[_f] = {'source': _src, 'p01': float(_q[0]), 'p50': float(_q[1]),
                     'p99': float(_q[2]), 'zero_frac': float((_v == 0).mean())}
        print(f"  {_f:<18} {_src[:22]:<22} {_q[0]:>12.4g} {_q[1]:>12.4g} {_q[2]:>12.4g} "
              f"{(_v == 0).mean() * 100:>5.1f}%")
    AUDIT['units'][_name] = _rows

def _ratio_check(name, X, num_fn, den_fn, label, expect):
    """Median of num/den over rows where both are positive; prints vs expected factor."""
    num = num_fn(X); den = den_fn(X)
    m = (num > 0) & (den > 0) & np.isfinite(num) & np.isfinite(den)
    if m.sum() < 100:
        print(f'  [{name}] {label}: too few valid rows'); return None
    r = num[m] / den[m]
    med = float(np.median(r)); iqr = [float(np.percentile(r, 25)), float(np.percentile(r, 75))]
    print(f'  [{name}] {label}: median ratio {med:.4g} (IQR {iqr[0]:.3g}–{iqr[1]:.3g}), '
          f'expected ≈ {expect}  (n={int(m.sum()):,})')
    return {'check': label, 'median': med, 'iqr': iqr, 'expected': expect, 'n': int(m.sum())}

I = SEM_IDX
AUDIT['unit_checks'] = []
print('\nUNIT CONSISTENCY CHECKS (derived from the data, not from documentation)')
for _name, _d in loader.datasets.items():
    _X = _d['X'].astype(np.float64)
    if _name in ('CICIDS-2017', 'CIC-BCCC-NRC-TabularIoTAttacks-2024'):
        # Flow Bytes/s == total bytes / (Flow Duration in seconds)  -> ratio 1 if duration is us
        r = _ratio_check(_name, _X,
            lambda X: (X[:, I['byte_count_fwd']] + X[:, I['byte_count_bwd']]),
            lambda X: X[:, I['byte_rate']] * X[:, I['flow_duration']],
            'total_bytes / (byte_rate * flow_duration)', '1e-6 if duration in us, 1 if in s')
        if r: AUDIT['unit_checks'].append({'dataset': _name, **r})
        r = _ratio_check(_name, _X,
            lambda X: X[:, I['flow_duration']],
            lambda X: X[:, I['iat_mean']] * np.maximum(X[:, I['pkt_count_fwd']] + X[:, I['pkt_count_bwd']] - 1, 1),
            'flow_duration / (iat_mean * (n_pkts-1))', '1 if duration and IAT share a unit')
        if r: AUDIT['unit_checks'].append({'dataset': _name, **r})
    if _name == 'UNSW-NB15':
        r = _ratio_check(_name, _X,
            lambda X: X[:, I['fwd_byte_rate']],
            lambda X: X[:, I['byte_count_fwd']] / np.maximum(X[:, I['flow_duration']], 1e-9),
            'sload / (sbytes / dur)', '8 if sload is bits/s and dur seconds')
        if r: AUDIT['unit_checks'].append({'dataset': _name, **r})
        r = _ratio_check(_name, _X,
            lambda X: X[:, I['fwd_iat_mean']] * np.maximum(X[:, I['pkt_count_fwd']] - 1, 1),
            lambda X: X[:, I['flow_duration']],
            '(sinpkt * (spkts-1)) / dur', '1000 if sinpkt is ms and dur seconds')
        if r: AUDIT['unit_checks'].append({'dataset': _name, **r})
    if _name == 'CICIoMT-2024':
        # IAT in CICIoT2023-family extractors: report its scale; values ~1e7–1e8 behave
        # like absolute timestamps rather than inter-arrival intervals.
        _v = _X[:, I['iat_mean']]; _v = _v[_v > 0]
        if len(_v):
            print(f'  [{_name}] iat_mean range: p01={np.percentile(_v, 1):.4g} '
                  f'p50={np.median(_v):.4g} p99={np.percentile(_v, 99):.4g}')
            AUDIT['unit_checks'].append({'dataset': _name, 'check': 'iat_mean scale',
                                         'p01': float(np.percentile(_v, 1)),
                                         'p50': float(np.median(_v)),
                                         'p99': float(np.percentile(_v, 99))})

## C. Window labels: ties, purity, and window length
Reproduces the current pipeline (balance → per-dataset 70/10/20 split → windows) and,
per dataset, reports how often a 32-record window is an exact 16/16 tie (currently
labelled benign), how pure windows are, and how this changes for W = 31 and 33.
Answers Reviewer 3 §3.5.

In [ ]:
def _window_label_stats(y_rows, W, S):
    n = (len(y_rows) - W) // S + 1 if len(y_rows) >= W else 0
    if n <= 0: return None
    idx = np.arange(n)[:, None] * S + np.arange(W)[None, :]
    frac = y_rows[idx].mean(axis=1)
    return {'n_windows': int(n),
            'tie_frac': float((np.abs(frac - 0.5) < 1e-9).mean()),
            'attack_window_frac': float((frac > 0.5).mean()),
            'mean_purity': float(np.maximum(frac, 1 - frac).mean()),
            'pure_frac': float(((frac == 0) | (frac == 1)).mean()),
            'minority_attack_frac': float(((frac > 0) & (frac <= 0.5)).mean())}

_rng = np.random.RandomState(42)
AUDIT['window_labels'] = {}
AUDIT['split_rows'] = {}
_SPLIT = {}
for _name, _d in loader.datasets.items():
    Xr = _d['X'].copy(); yr = _d['y'].copy(); cr = _d['ctx'].copy()
    Xr, yr, cr = loader._balance(Xr, yr, cr, Config.TARGET_ATK_BEN_RATIO, _rng, _name)
    n = len(yr); n_trva = int(n * 0.8); n_va = int(n * Config.VAL_FRAC)
    parts = {'train': slice(0, n_trva - n_va), 'val': slice(n_trva - n_va, n_trva),
             'test': slice(n_trva, n)}
    _SPLIT[_name] = {k: (Xr[s], yr[s]) for k, s in parts.items()}
    AUDIT['split_rows'][_name] = {k: int(len(yr[s])) for k, s in parts.items()}
    AUDIT['window_labels'][_name] = {}
    print(f'\n{_name}: rows train/val/test = '
          + '/'.join(f"{AUDIT['split_rows'][_name][k]:,}" for k in ('train', 'val', 'test')))
    for W in (31, 32, 33):
        st = _window_label_stats(yr[parts['test']], W, Config.STRIDE)
        AUDIT['window_labels'][_name][f'W{W}_test'] = st
        if st:
            print(f"  W={W} test: windows={st['n_windows']:,}  ties={st['tie_frac']*100:.2f}%  "
                  f"attack-windows={st['attack_window_frac']*100:.1f}%  "
                  f"mean purity={st['mean_purity']:.3f}  pure={st['pure_frac']*100:.1f}%  "
                  f"benign-labelled windows containing attacks={st['minority_attack_frac']*100:.1f}%")
    st = _window_label_stats(yr[parts['train']], 32, Config.STRIDE)
    AUDIT['window_labels'][_name]['W32_train'] = st

## D. Exact-duplicate audit across all partitions and datasets
Hashes **every** canonical row (not a sample) before scaling and counts identical
vectors within each partition, across train/validation/test, and across datasets.
Answers Reviewer 3 §3.7.

In [ ]:
import hashlib
def _hashes(X):
    Xc = np.ascontiguousarray(X.astype(np.float32))
    return np.array([hashlib.blake2b(r.tobytes(), digest_size=12).hexdigest() for r in Xc])

AUDIT['duplicates'] = {}
_all_hash = {}
for _name, parts in _SPLIT.items():
    H = {k: _hashes(v[0]) for k, v in parts.items()}
    _all_hash[_name] = np.concatenate(list(H.values()))
    res = {}
    for k, h in H.items():
        res[f'{k}_rows'] = int(len(h))
        res[f'{k}_within_dup_frac'] = float(1 - len(np.unique(h)) / max(len(h), 1))
    tr = set(H['train'])
    res['test_rows_in_train_frac'] = float(np.mean([x in tr for x in H['test']])) if len(H['test']) else 0.0
    res['val_rows_in_train_frac'] = float(np.mean([x in tr for x in H['val']])) if len(H['val']) else 0.0
    va = set(H['val'])
    res['test_rows_in_val_frac'] = float(np.mean([x in va for x in H['test']])) if len(H['test']) else 0.0
    # label conflicts: identical vectors with different labels (inside train)
    _yt = parts['train'][1]
    _df = pd.DataFrame({'h': H['train'], 'y': _yt})
    _g = _df.groupby('h')['y'].nunique()
    res['train_conflicting_label_vectors'] = int((_g > 1).sum())
    AUDIT['duplicates'][_name] = res
    print(f"{_name}:")
    for k, v in res.items():
        print(f"   {k:<34} {v:.4f}" if isinstance(v, float) else f"   {k:<34} {v:,}")
_names = list(_all_hash)
AUDIT['cross_dataset_shared_vectors'] = {}
for i in range(len(_names)):
    for j in range(i + 1, len(_names)):
        a, b = set(_all_hash[_names[i]]), set(_all_hash[_names[j]])
        n_sh = len(a & b)
        AUDIT['cross_dataset_shared_vectors'][f'{_names[i]} | {_names[j]}'] = n_sh
        print(f"  shared identical vectors {_names[i][:20]} | {_names[j][:20]}: {n_sh:,}")

## E. Save
Download `bridge_audit/bridge_audit.json` (Output tab → bridge_audit) and send it back.

In [ ]:
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o
with open(os.path.join(AUDIT_OUT, 'bridge_audit.json'), 'w') as f:
    json.dump(_clean(AUDIT), f, indent=2)
print('Saved', os.path.join(AUDIT_OUT, 'bridge_audit.json'))